# TuiML Hub Integration

This tutorial covers TuiML's Hub system for:
- **Local Registry** - Register and discover local components
- **Remote Hub** - Browse, download, and use algorithms from the community hub

The Hub enables sharing algorithms and building a community-driven ML ecosystem.

## 1. Local Registry

The registry manages local TuiML components and provides discovery capabilities.

### 1.1 Browsing Registered Components

In [1]:
from tuiml.hub import registry
from tuiml.hub.types import ComponentType

# Import algorithms to trigger hub registration via decorators
import tuiml.algorithms

# List all registered classifiers
classifiers = registry.list_names(ComponentType.CLASSIFIER)
print(f"Registered classifiers: {len(classifiers)}")
print(f"First 10: {classifiers[:10]}")

Registered classifiers: 42
First 10: ['NaiveBayesClassifier', 'NaiveBayesMultinomialClassifier', 'BayesianNetworkClassifier', 'DecisionStumpClassifier', 'C45TreeClassifier', 'RandomTreeClassifier', 'RandomForestClassifier', 'ReducedErrorPruningTreeClassifier', 'HoeffdingTreeClassifier', 'LogisticModelTreeClassifier']


In [2]:
# List regressors
regressors = registry.list_names(ComponentType.REGRESSOR)
print(f"\nRegistered regressors: {len(regressors)}")
print(f"Examples: {regressors[:5]}")


Registered regressors: 20
Examples: ['GaussianProcessesRegressor', 'M5ModelTreeRegressor', 'LocallyWeightedLearningRegressor', 'LinearRegression', 'SimpleLinearRegression']


In [3]:
# List clusterers
clusterers = registry.list_names(ComponentType.CLUSTERER)
print(f"\nRegistered clusterers: {len(clusterers)}")
print(f"Examples: {clusterers[:5]}")


Registered clusterers: 8
Examples: ['KMeansClusterer', 'FarthestFirstClusterer', 'AgglomerativeClusterer', 'DBSCANClusterer', 'GaussianMixtureClusterer']


### 1.2 Getting Component Information

In [4]:
# Get details about a specific component
info = registry.get_info("RandomForestClassifier")

print("RandomForestClassifier Component Info:")
print(f"  Name: {info.get('name')}")
print(f"  Type: {info.get('type')}")
print(f"  Version: {info.get('version')}")
print(f"  Tags: {info.get('tags', [])}")

RandomForestClassifier Component Info:
  Name: RandomForestClassifier
  Type: classifier
  Version: 1.0.0
  Tags: ['trees', 'ensemble', 'random', 'bagging']


In [5]:
# Search for components by name or tag
results = registry.search("forest")
print(f"Search results for 'forest': {[r['name'] for r in results]}")

results = registry.search("bayes")
print(f"Search results for 'bayes': {[r['name'] for r in results]}")

Search results for 'forest': ['DecisionStumpClassifier', 'C45TreeClassifier', 'RandomTreeClassifier', 'RandomForestClassifier', 'HoeffdingTreeClassifier', 'LogisticModelTreeClassifier', 'BaggingClassifier', 'RandomSubspaceClassifier', 'IsolationForestDetector', 'LocalOutlierFactorDetector', 'EllipticEnvelopeDetector', 'OneClassSVMDetector', 'ABODDetector']
Search results for 'bayes': ['NaiveBayesClassifier', 'NaiveBayesMultinomialClassifier', 'BayesianNetworkClassifier', 'GaussianProcessesRegressor', 'HoeffdingTreeClassifier', 'LocallyWeightedLearningRegressor', 'VotingClassifier', 'StackingClassifier', 'RegressionByDiscretization', 'CatBoostClassifier', 'CatBoostRegressor', 'Prophet']


### 1.3 Creating Components from Registry

In [6]:
from tuiml.datasets import load_iris
from tuiml.evaluation import train_test_split, accuracy_score

# Create a component by name
model = registry.create("RandomForestClassifier", n_estimators=10, random_state=42)

# Use it normally
X, y = load_iris()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Model created via registry:")
print(f"  Type: {type(model).__name__}")
print(f"  Accuracy: {accuracy_score(y_test, y_pred):.2%}")

Model created via registry:
  Type: RandomForestClassifier
  Accuracy: 100.00%


### 1.4 Registering Custom Components

In [7]:
import numpy as np

# Register a custom classifier
@registry.register(
    "classifier",
    name="MajorityClassifier",
    tags=["baseline", "simple"],
    version="1.0.0",
    author="TuiML Tutorial"
)
class MajorityClassifier:
    """A simple baseline classifier that always predicts the majority class."""
    
    def __init__(self):
        self.majority_class_ = None
    
    def fit(self, X, y):
        """Learn the majority class."""
        unique, counts = np.unique(y, return_counts=True)
        self.majority_class_ = unique[np.argmax(counts)]
        return self
    
    def predict(self, X):
        """Predict majority class for all samples."""
        return np.full(len(X), self.majority_class_)

print("Custom classifier registered!")
search_results = registry.search("majority")
found = any(r["name"] == "MajorityClassifier" for r in search_results)
print(f"  Can find via search: {found}")

Custom classifier registered!
  Can find via search: True


In [8]:
# Use the registered custom classifier
baseline = registry.create("MajorityClassifier")
baseline.fit(X_train, y_train)
y_pred_baseline = baseline.predict(X_test)

print(f"Baseline Accuracy: {accuracy_score(y_test, y_pred_baseline):.2%}")

Baseline Accuracy: 30.00%


## 2. Remote Hub

The remote hub allows you to browse and download algorithms shared by the community.

### 2.1 Browsing Remote Algorithms

In [9]:
from tuiml.hub import remote

# Note: Requires a running TuiML Hub server
# Set TUIML_HUB_URL environment variable or it defaults to localhost:8000

try:
    # Browse available algorithms
    algorithms = remote.browse(limit=10)
    
    print(f"Available algorithms: {len(algorithms)}")
    for algo in algorithms:
        print(f"  - {algo.name}: {algo.description[:50]}...")
except ConnectionError as e:
    print(f"Hub not available: {e}")
    print("Run 'python -m tuiml_hub.backend.app' to start the hub server")

Available algorithms: 7
  - simple_decision_tree: A lightweight decision tree classifier optimized f...
  - MRMRSelector: Minimum Redundancy Maximum Relevance feature selec...
  - QuantileTransformer: Transform features to uniform or normal distributi...
  - TargetEncoder: Encode categorical features using target variable ...
  - SAINTClassifier: Self-Attention and Intersample Attention Transform...
  - NODEClassifier: Neural Oblivious Decision Ensembles - differentiab...
  - TabNetClassifier: Attentive interpretable tabular learning with sequ...


### 2.2 Searching for Algorithms

In [10]:
try:
    # Search for specific algorithms
    results = remote.search("SAINTClassifier")
    
    print(f"Search results for 'random forest': {len(results)}")
    for algo in results:
        print(f"  - {algo.name} by {algo.author}")
except (ConnectionError, RuntimeError):
    print("Hub not available or no results. Start the server to use remote features.")

Search results for 'random forest': 1
  - SAINTClassifier by tuiml


### 2.3 Installing Remote Algorithms

In [11]:
try:
    # Browse and install an algorithm that has source files
    algorithms = remote.browse(limit=10)
    installed_name = None

    for algo in algorithms:
        try:
            remote.install(algo.name, force=True)
            installed_name = algo.name
            break
        except ValueError:
            continue

    if installed_name:
        print(f"Algorithm '{installed_name}' installed successfully!")
        print(f"Installed to: {remote.cache_dir / installed_name}")
    else:
        print("No algorithms with source files available on the hub.")
except ConnectionError:
    print("Hub not available.")
except Exception as e:
    print(f"Installation error: {e}")

Installing 'simple_decision_tree' from hub...
Installing 'MRMRSelector' from hub...
  Downloaded README.md (4147 bytes)
  Downloaded algorithm.json (2388 bytes)
  Downloaded mrmr_selector.py (12175 bytes)
✓ Installed 'MRMRSelector' to /Users/nileshverma/.tuiml/algorithms/MRMRSelector
Algorithm 'MRMRSelector' installed successfully!
Installed to: /Users/nileshverma/.tuiml/algorithms/MRMRSelector


### 2.4 Using Remote Algorithms

In [12]:
try:
    # Load a previously installed algorithm (from cell above)
    if installed_name:
        RemoteAlgo = remote.load(installed_name)

        # Use it like any other TuiML algorithm
        model = RemoteAlgo()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        print(f"Remote algorithm accuracy: {accuracy_score(y_test, y_pred):.2%}")
    else:
        print("No algorithm installed to load.")
except Exception as e:
    print(f"Remote algorithm not available: {e}")

Remote algorithm not available: 'MRMRSelector' object has no attribute 'predict'


### 2.5 Convenient `use()` Method

In [13]:
try:
    # Combine install + load + instantiate in one call
    # Use a known algorithm with source files from the hub
    algorithms = remote.browse(limit=10)
    for algo in algorithms:
        try:
            model = remote.use(algo.name)
            model.fit(X_train, y_train)
            print(f"Model '{algo.name}' ready! Fitted successfully.")
            break
        except (ValueError, Exception):
            continue
    else:
        print("No usable remote algorithms found.")
except ConnectionError:
    print("Hub not available.")
except Exception as e:
    print(f"Remote algorithm not available: {e}")

Installing 'simple_decision_tree' from hub...
Model 'MRMRSelector' ready! Fitted successfully.


## 3. Algorithm Discovery Workflow

A typical workflow for finding and using algorithms.

In [14]:
# Step 1: Check local registry first
print("=== Local Registry ===")
local_classifiers = registry.list_names(ComponentType.CLASSIFIER)
print(f"Local classifiers available: {len(local_classifiers)}")

# Step 2: Search for what you need
search_term = "forest"
local_results = registry.search(search_term)
# Filter to components with search term in the name
name_matches = [r for r in local_results if search_term.lower() in r["name"].lower()]
print(f"\nLocal search for '{search_term}': {[r['name'] for r in name_matches]}")

# Step 3: Create and use
if name_matches:
    algo_name = name_matches[0]["name"]
    model = registry.create(algo_name, n_estimators=10, random_state=42)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"\n{algo_name} accuracy: {acc:.2%}")

=== Local Registry ===
Local classifiers available: 43

Local search for 'forest': ['RandomForestClassifier', 'IsolationForestDetector']

RandomForestClassifier accuracy: 100.00%


In [15]:
# Step 4: If not found locally, check remote hub
print("\n=== Remote Hub ===")
try:
    remote_results = remote.search("custom ensemble")
    print(f"Remote search results: {len(remote_results)}")
    
    # Install and use if found
    if remote_results:
        algo = remote_results[0]
        model = remote.use(algo.name)
        model.fit(X_train, y_train)
        print(f"{algo.name} accuracy: {accuracy_score(y_test, model.predict(X_test)):.2%}")
except (ConnectionError, RuntimeError):
    print("Remote hub not available - using local algorithms only.")


=== Remote Hub ===
Remote search results: 0


## 4. Component Types Reference

Available component types in TuiML Hub.

In [16]:
from tuiml.hub.types import ComponentType

print("Available Component Types:")
for ct in ComponentType:
    count = len(registry.list(ct))
    print(f"  - {ct.value}: {count} components")

Available Component Types:
  - algorithm: 0 components
  - classifier: 43 components
  - clusterer: 8 components
  - regressor: 20 components
  - associator: 3 components
  - preprocessor: 0 components
  - filter: 0 components
  - transformer: 0 components
  - feature_selector: 0 components
  - feature_extractor: 0 components
  - feature_constructor: 0 components
  - metric: 0 components
  - evaluator: 0 components
  - timeseries: 0 components


## 5. Dataset Hub

TuiML Hub also hosts community datasets. Browse, search, and load datasets directly into your workflows.

### 5.1 Browsing Remote Datasets

In [17]:
from tuiml.hub import datasets

# Note: Requires a running TuiML Hub server
# Set TUIML_HUB_URL environment variable or it defaults to localhost:8000

try:
    # Browse available datasets
    all_datasets = datasets.browse(limit=10)
    
    print(f"Available datasets: {len(all_datasets)}")
    for ds in all_datasets:
        print(f"  - {ds.name}: {ds.rows:,} rows × {ds.columns} cols ({ds.task_type})")
except ConnectionError as e:
    print(f"Hub not available: {e}")
    print("Run 'bash run.sh' in tuiml-hub/ to start the server")

Available datasets: 3
  - customer-segments: 2,000 rows × 8 cols (clustering)
  - bike-sharing: 17,389 rows × 17 cols (regression)
  - wine-quality: 6,497 rows × 13 cols (classification)


### 5.2 Searching Datasets

In [18]:
try:
    # Search for specific datasets
    results = datasets.search("wine")
    
    print(f"Search results for 'wine': {len(results)}")
    for ds in results:
        print(f"  - {ds.name}: {ds.description[:60]}...")
except ConnectionError:
    print("Hub not available. Start the server to use remote features.")

Search results for 'wine': 1
  - wine-quality: Combined red and white wine quality dataset with physicochem...


### 5.3 Getting Dataset Info

In [19]:
try:
    # Get detailed info about a specific dataset
    info = datasets.info("wine-quality")
    
    print(f"Dataset: {info.display_name}")
    print(f"  Description: {info.description}")
    print(f"  Task: {info.task_type}")
    print(f"  Size: {info.rows:,} rows × {info.columns} columns")
    print(f"  Format: {info.format}")
    print(f"  Downloads: {info.downloads}")
except ConnectionError:
    print("Hub not available.")
except Exception as e:
    print(f"Dataset not found: {e}")

Dataset: Wine Quality
  Description: Combined red and white wine quality dataset with physicochemical properties and quality ratings
  Task: classification
  Size: 6,497 rows × 13 columns
  Format: csv
  Downloads: 344


### 5.4 Loading Datasets

In [20]:
try:
    # Load a dataset directly as a pandas DataFrame
    df = datasets.load("wine-quality")
    
    print(f"Loaded dataset: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"\nFirst 5 rows:")
    print(df.head())
    
    # Datasets are cached locally in ~/.tuiml/datasets/
    print(f"\nCache location: ~/.tuiml/datasets/wine-quality/")
except ConnectionError:
    print("Hub not available.")
except Exception as e:
    print(f"Could not load dataset: {e}")

Loaded dataset: 10 rows × 13 columns

First 5 rows:
   fixed_acidity  volatile_acidity  citric_acid  residual_sugar  chlorides  \
0            7.4              0.70         0.00             1.9      0.076   
1            7.8              0.88         0.00             2.6      0.098   
2            7.8              0.76         0.04             2.3      0.092   
3           11.2              0.28         0.56             1.9      0.075   
4            7.4              0.70         0.00             1.9      0.076   

   free_sulfur_dioxide  total_sulfur_dioxide  density    pH  sulphates  \
0                 11.0                  34.0   0.9978  3.51       0.56   
1                 25.0                  67.0   0.9968  3.20       0.68   
2                 15.0                  54.0   0.9970  3.26       0.65   
3                 17.0                  60.0   0.9980  3.16       0.58   
4                 11.0                  34.0   0.9978  3.51       0.56   

   alcohol type  quality  
0      

### 5.5 Using Hub Datasets with TuiML

Hub datasets integrate seamlessly with TuiML's training APIs.

In [21]:
import tuiml

try:
    # Load hub dataset and train a model
    df = datasets.load("wine-quality")
    
    # Use with tuiml.train()
    result = tuiml.train(
        "RandomForestClassifier",
        df,
        target="quality",
        cv=5
    )
    print(f"Accuracy on wine-quality: {result.metrics['cv_accuracy_score_mean']:.4f}")
except Exception as e:
    print(f"Example requires hub server: {e}")

Accuracy on wine-quality: 0.5000


### 5.6 Dataset CLI Commands

You can also manage datasets from the command line:

```bash
# List available datasets
tuiml datasets list

# List classification datasets
tuiml datasets list --task classification

# Get detailed info about a dataset
tuiml datasets info wine-quality

# Download a dataset to local cache
tuiml datasets download wine-quality

# Download to a specific directory
tuiml datasets download wine-quality -o ./my_data/
```

**To upload datasets**, see `community/02_hub_publish` tutorial for detailed instructions on publishing datasets via CLI and Python.